In [ ]:
# ==========================================================
# RANDOM SPLIT INTO 22 GROUPS (A > 0)
# ==========================================================

import os
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------

INPUT_FILE = "Nuclei.xlsx"
OUTPUT_DIR = "splits"
OUTPUT_FILE = f"{OUTPUT_DIR}/random_22_groups.xlsx"

N_GROUPS = 22
SEED = 42   # for reproducibility

# ----------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------

df = pd.read_excel(INPUT_FILE)[["Z", "N", "A"]].copy()


df = df[df["A"] > 0 ].reset_index(drop=True)

print("===================================================")
print("RANDOM SPLIT INTO 22 GROUPS (A > 0)")
print("===================================================")
print(f"Total nuclei with A > 0: {len(df)}")
print("---------------------------------------------------")

# ----------------------------------------------------------
# RANDOM SHUFFLE
# ----------------------------------------------------------

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(df))
df = df.loc[perm].reset_index(drop=True)

# ----------------------------------------------------------
# ASSIGN GROUPS EVENLY
# ----------------------------------------------------------

group_ids = np.zeros(len(df), dtype=int)

sizes = [len(df) // N_GROUPS] * N_GROUPS
for i in range(len(df) % N_GROUPS):
    sizes[i] += 1

start = 0
for g, sz in enumerate(sizes, start=1):
    group_ids[start:start+sz] = g
    start += sz

df["group"] = group_ids

# ----------------------------------------------------------
# SAVE
# ----------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
df[["Z", "N", "group"]].to_excel(OUTPUT_FILE, index=False)

# ----------------------------------------------------------
# PRINT OUTPUT SUMMARY
# ----------------------------------------------------------

print("\nGroup sizes:")
print(df["group"].value_counts().sort_index())

print("\nSaved splits to:", OUTPUT_FILE)
print("===================================================")


In [ ]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from math import sqrt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras import layers, regularizers, models

# ---------------- Reproducibility ----------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------- Load data ----------------
df = pd.read_excel("Nuclei.xlsx").copy()

groups_df = pd.read_excel("splits/random_22_groups.xlsx")

df = df.merge(
    groups_df,
    on=["Z", "N"],
    how="left",
    validate="one_to_one"
)

# ---------------- Bin definition (original 4 bins) ----------------
def assign_bin(x):
    if x <= -0.5: return 1
    elif x <= 0.0: return 2
    elif x <= 0.5: return 3
    else: return 4

# ---------------- Feature columns ----------------
feature_cols = [c for c in df.columns if c not in ["deltaM","group"]]

# ---------------- Deep NN (same as original) ----------------
def build_deep_nn():
    model = models.Sequential([
        layers.Input(shape=(len(feature_cols),)),
        layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.1)),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# ---------------- Save function ----------------
def save_function(root, g, step, b, model, scaler):
    path = f"{root}/g-{g}/step{step}_bin{b}"
    os.makedirs(path, exist_ok=True)
    model.save(f"{path}/model.keras")
    np.save(f"{path}/scaler_mean.npy", scaler.mean_)
    np.save(f"{path}/scaler_scale.npy", scaler.scale_)

# ============================================================
# MAIN LOOP OVER GROUPS
# ============================================================

MODEL_ROOT = "models_retry_adaptive_bins_all_features"
os.makedirs(MODEL_ROOT, exist_ok=True)

for TEST_GROUP in range(1, 23):

    print("\n" + "="*60)
    print(f"RETRY MAJOR RUN — TEST GROUP g-{TEST_GROUP}")
    print("="*60)

    train_mask = df["group"] != TEST_GROUP
    test_mask  = df["group"] == TEST_GROUP

    print(f"Number of test nuclei (g-{TEST_GROUP}): {test_mask.sum()}")

    # ---------- Initial residual ----------
    current_residual = df["deltaM"].values.copy()
    current_bins = np.array([assign_bin(x) for x in current_residual])

    # ---------- STEPWISE TRAINING (4 cycles only) ----------
    for step in [1,2,3,4]:

        print(f"\n=========== STEP {step} ===========")
        learned = np.zeros(len(df))

        for b in [1,2,3,4]:

            idx = np.where((current_bins == b) & train_mask)[0]
            if len(idx) == 0:
                continue

            X = df.loc[idx, feature_cols].values
            y = current_residual[idx]

            scaler = StandardScaler()
            Xs = scaler.fit_transform(X)

            model = build_deep_nn()
            model.fit(Xs, y, epochs=1000, batch_size=32, verbose=0)

            learned[idx] = model.predict(Xs, verbose=0).flatten()

            save_function(MODEL_ROOT, TEST_GROUP, step, b, model, scaler)

        # --------- Update residual ---------
        current_residual = current_residual - learned

        # --------- Adaptive bin update ---------
        current_bins = np.array([assign_bin(x) for x in current_residual])

        rmse_train = sqrt(mean_squared_error(
            current_residual[train_mask],
            np.zeros(np.sum(train_mask))
        ))
        print(f"TRAIN RMSE after step {step}: {rmse_train:.6f}")

    # ---------- ORACLE TEST EVALUATION ----------
    results = []

    for _, row in df[test_mask].iterrows():

        X_input = row[feature_cols].values.reshape(1,-1)
        r = row.deltaM

        for step in [1,2,3,4]:

            b = assign_bin(r)

            try:
                model = tf.keras.models.load_model(
                    f"{MODEL_ROOT}/g-{TEST_GROUP}/step{step}_bin{b}/model.keras"
                )
                sc = StandardScaler()
                sc.mean_ = np.load(f"{MODEL_ROOT}/g-{TEST_GROUP}/step{step}_bin{b}/scaler_mean.npy")
                sc.scale_ = np.load(f"{MODEL_ROOT}/g-{TEST_GROUP}/step{step}_bin{b}/scaler_scale.npy")
                corr = model.predict(sc.transform(X_input), verbose=0)[0,0]
            except:
                corr = 0.0

            r = r - corr

            if step == 1: residual_1 = r
            if step == 2: residual_2 = r
            if step == 3: residual_3 = r
            if step == 4: residual_4 = r

        results.append({
            "Z": row.Z,
            "N": row.N,
            "A": row.A,
            "residual_1": residual_1,
            "residual_2": residual_2,
            "residual_3": residual_3,
            "residual_4": residual_4,
        })

    df_res = pd.DataFrame(results)

    print("\nPer-nucleus residuals (sorted by |residual_4|):")

    df_print = (
        df_res.assign(abs_r4=lambda d: d["residual_4"].abs())
              .sort_values("abs_r4", ascending=False)
              .drop(columns="abs_r4")
    )

    print(df_print.to_string(index=False))

    rmse_4 = sqrt(mean_squared_error(df_res["residual_4"], np.zeros(len(df_res))))
    print(f"FINAL TEST RMSE (g-{TEST_GROUP}, residual_4) = {rmse_4:.6f} MeV")

    os.makedirs("results", exist_ok=True)
    out_path = f"results/test_g-{TEST_GROUP}_retry_stepwise_residuals_r4.xlsx"
    df_res.to_excel(out_path, index=False)
    print(f"Saved to {out_path}")

print("\nDONE — adaptive bins + 4 residual cycles complete.")
